# CreaseIQ — LangGraph Workflow (v1.6 architecture)

The workflow now lives in the `creaseiq` package (`backend/src/creaseiq/`); this notebook imports it, builds the graph, and runs demos and branch tests. **All settings come from `backend/config/`** (nothing tunable is hard-coded in the code).

| File | Contents |
|---|---|
| `config/creaseiq.toml` | switches, limits, retrieval thresholds, LLM endpoint, guardrail thresholds |
| `config/lexicon.toml` | word lists, regex patterns, format → tool map, stub-model data, mock web results |
| `config/prompts.toml` | system prompts and templates |
| `config/strings.toml` | fixed user-facing messages (English, Hindi) |
| `src/kb_config.toml` | knowledge base (chunking, BM25, Chroma, RRF) |

| Module (`src/creaseiq/`) | Role |
|---|---|
| `config.py` | loads the TOML files; run-time `FLAGS` and test hooks `SIM` |
| `state.py` | `CreaseState`, `RouteResult`, `Analysis` |
| `llm.py` | stub or OpenAI-compatible model; `llm_text`, `llm_structured` |
| `tools.py` | per-format knowledge-base search (hybrid BM25 + semantic via `build_kb.py`), `get_clause`, mock `web_search` |
| `route.py` | `load_or_initialize_state`, `decide_route`, `understand_and_route` |
| `rag.py` | retrieve → evidence check → reformulate → analyse → answer → validate → repair |
| `nodes.py` | gates, web branch, clarification, terminal handlers |
| `guardrails.py` | `sanitize_input`, `input_guard`, `output_guard`, output checks |
| `routers.py`, `graph.py` | the diamonds and the compiled `StateGraph` |
| `runner.py`, `branch_tests.py` | `ask()` (FastAPI stand-in) and the branch tests |

Set `llm.use_stub = false` in `creaseiq.toml` to use the local Ollama model. Build the knowledge base first: `.venv/bin/python src/build_kb.py build`. Run the tests without Jupyter: `cd src && ../.venv/bin/python -m creaseiq.branch_tests`.

## 1. Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))      # run from backend/src

from creaseiq.config import settings, FLAGS, SIM
from creaseiq.graph import build_graph, NODES, GUARDRAIL_ADDITIONS
from creaseiq.runner import ask
from creaseiq.branch_tests import run_all

print("stub LLM:", settings.llm.use_stub, "| general Q&A:", FLAGS["general_qa_enabled"])

## 2. Build the graph
PII redaction is **not** in the graph: it runs in FastAPI before the graph is entered (`sanitize_input` stands in for it).

In [ ]:
graph = build_graph()
print(len(NODES), "nodes compiled")

In [ ]:
# Check the node set against the architecture diagram (RETRIEVE, RT, EVIDENCE, REFORM, ANALYSE, RENDER form the RAG box)
EXPECTED = {"load_or_initialize_state", "understand_and_route", "general_qa_gate", "web_search", "answer_general_question", "handle_non_cricket",
            "scope_gate", "handle_unsupported", "context_gate", "clarification_gate", "ask_clarification", "save_clarification_state",
            "retrieve_rules", "retrieval_tools", "check_evidence", "reformulate_query", "analyse_scenario", "generate_answer",
            "validate_final_answer", "constrained_repair", "save_final_state", "handle_insufficient_evidence", "handle_error"}
actual = set(graph.get_graph().nodes) - {"__start__", "__end__"}
assert actual == EXPECTED | GUARDRAIL_ADDITIONS, (actual ^ (EXPECTED | GUARDRAIL_ADDITIONS))
print("graph nodes match the architecture diagram (+ guardrail additions)")
print(graph.get_graph().draw_mermaid())      # paste into any Mermaid viewer to see the compiled graph

## 3. Run it
`ask()` redacts contact PII, invokes the graph on a `thread_id` (the conversation id) and prints the node path it took.

In [ ]:
_ = ask(graph, "Is it a no ball if the bowler's front foot lands beyond the popping crease in a T20I?", "demo-1")

In [ ]:
# Clarification: the first turn asks for the format, the second turn (same thread) answers.
_ = ask(graph, "The batter left the crease and the ball hit the stumps. Is he out?", "demo-2")
_ = ask(graph, "It was an ODI and the ball was live.", "demo-2")

## 4. Test every branch
Each scenario runs on its own thread and is checked against the expected status (cases and failure-injection hooks are in `creaseiq/branch_tests.py`). Retrieval-dependent cases (`retry_then_answer`, `no_passage_abstain`, `outside_sources`) were written for the old mock corpus and need re-tuning against the real knowledge base.

In [ ]:
results = run_all(graph)
failed = [r[0] for r in results if not r[1]]
print("\nfailed:", failed or "none")

In [ ]:
# Look at one run in detail: the repair path (the first answer cites a clause that was never retrieved)
SIM["bad_clause_once"] = True
r = ask(graph, "Is it a no ball if the bowler's front foot lands beyond the popping crease in a T20I?", "detail-repair")
print("validation issues after the repair:", r["values"]["validation_issues"], "| repair_attempt =", r["values"]["repair_attempt"])

## Next steps
1. Extract `clause_title` / `refers_outside` at knowledge-base build time and re-tune `retrieval.min_score` and the three retrieval-dependent cases.
2. Set `llm.use_stub = false` and check `understand_and_route` structured output on Ollama, then Sarvam (English and Hindi).
3. Swap `InMemorySaver` for the production checkpointer, and move `sanitize_input` into FastAPI.
4. Replace the mock `web_search` with the chosen provider.